In [37]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler, RobustScaler, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

In [38]:
X, y = make_regression(n_samples=60, n_features=20, n_informative=8, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [39]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_regression)),
    ('model', LinearRegression())
])

In [40]:
multi_model_param_grid = [
    {
        'selector__k': [5, 8, 12, 'all'],
        'model': [LinearRegression()],
        'scaler': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough']
    },
    {
        'selector__k': [5, 8, 12, 'all'],
        'model': [Ridge()],
        'scaler': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'],
        'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
    },
    {
        'selector__k': [5, 8, 12, 'all'],
        'model': [Lasso()],
        'scaler': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'],
        'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
    },
    {
        'selector__k': [5, 8, 12, 'all'],
        'model': [RandomForestRegressor(random_state=42)],
        'scaler': ['passthrough'],
        'model__max_depth': [3, 5, 8, 12, None],
        'model__min_samples_leaf': [1, 2, 4]
    },
    {
        'selector__k': [5, 8, 12, 'all'],
        'model': [GradientBoostingRegressor(random_state=42)],
        'scaler': ['passthrough'],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__subsample': [0.8, 1.0]
    }
]

In [41]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=multi_model_param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)

In [42]:
print(f"Training grid search across multiple algorithms...")
grid_search.fit(X_train, y_train)

print("\n=== OPTIMAL CONFIGURATION ===")
print(f"Best Model/Parameters: {grid_search.best_params_}")

Training grid search across multiple algorithms...

=== OPTIMAL CONFIGURATION ===
Best Model/Parameters: {'model': Lasso(), 'model__alpha': 1.0, 'scaler': 'passthrough', 'selector__k': 'all'}


In [43]:
results = pd.DataFrame(grid_search.cv_results_)

results['model_name'] = results['param_model'].apply(lambda x: x.__class__.__name__)

results['selected_k'] = results['param_selector__k']

leaderboard = results[['model_name', 'param_scaler', 'selected_k', 'mean_test_score', 'std_test_score', 'rank_test_score']]

leaderboard['rmse_score'] = -leaderboard['mean_test_score']
leaderboard = leaderboard.drop(columns=['mean_test_score']).sort_values(by='rank_test_score')

print("--- MULTI-MODEL CROSS-VALIDATION LEADERBOARD ---")
print(leaderboard.to_string(index=False))

# best_overall_pipeline = grid_search.best_estimator_
# print(f"\nWinning Model Architecture: {best_overall_pipeline}")

--- MULTI-MODEL CROSS-VALIDATION LEADERBOARD ---
               model_name     param_scaler selected_k  std_test_score  rank_test_score   rmse_score
                    Lasso      passthrough        all       64.226626                1   133.386890
                    Lasso StandardScaler()        all       60.985898                2   137.392145
                    Lasso   RobustScaler()        all       44.397095                3   139.397762
                    Lasso   MinMaxScaler()        all      109.333202                4   179.850221
                    Lasso   RobustScaler()        all      165.071945                5   235.257697
                    Lasso      passthrough        all      176.170921                6   242.458874
                    Lasso StandardScaler()        all      177.788354                7   245.355657
                    Lasso   MinMaxScaler()        all      197.998471                8   263.439219
                    Ridge   RobustScaler()        a

C:\Users\thoma\AppData\Local\Temp\ipykernel_9408\4254125664.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leaderboard['rmse_score'] = -leaderboard['mean_test_score']


In [44]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== HELD-OUT TEST METRICS ===")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r2:.4f}")


=== HELD-OUT TEST METRICS ===
Mean Absolute Error (MAE): 11.4172
Mean Squared Error (MSE): 173.7084
Root Mean Squared Error (RMSE): 13.1798
R-squared (R2): 0.9945
